## Project 1

### Eliminating Child Care Deserts in New York State through Optimization


In [11]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

In [12]:
# 1) Read data files
avg_individual_income = pd.read_csv('ChildCareDeserts_Data/avg_individual_income.csv')
avg_individual_income.head()

child_care_regulated = pd.read_csv('ChildCareDeserts_Data/child_care_regulated.csv')
child_care_regulated.head()

employment_rate = pd.read_csv('ChildCareDeserts_Data/employment_rate.csv')
employment_rate.head()

population = pd.read_csv('ChildCareDeserts_Data/population.csv')
population.head()

potential_locations = pd.read_csv('ChildCareDeserts_Data/potential_locations.csv')
potential_locations.head()

,zipcode,latitude,longitude
0,501,40.816376,-73.040796
1,501,40.817455,-73.044202
2,501,40.813873,-73.042182
3,501,40.814152,-73.037265
4,501,40.824673,-73.047431


In [13]:
# 2) Standardize ZIP column names to `zipcode`:
#    - income: rename 'ZIP code' -> 'zipcode'
#    - childcare: rename 'zip_code' -> 'zipcode'
#    - ensure others are already 'zipcode' or rename accordingly

# Print current column names for each dataframe
print("Income columns:", avg_individual_income.columns.tolist())
print("Childcare columns:", child_care_regulated.columns.tolist())
print("Employment columns:", employment_rate.columns.tolist())
print("Population columns:", population.columns.tolist())
print("Potential locations columns:", potential_locations.columns.tolist())

# Standardize ZIP code columns
avg_individual_income = avg_individual_income.rename(columns={'ZIP code': 'zipcode'})
child_care_regulated = child_care_regulated.rename(columns={'zip_code': 'zipcode'})

# Verify the changes
print("\nAfter standardization:")
print("Income columns:", avg_individual_income.columns.tolist())
print("Childcare columns:", child_care_regulated.columns.tolist())
print("Employment columns:", employment_rate.columns.tolist())
print("Population columns:", population.columns.tolist())
print("Potential locations columns:", potential_locations.columns.tolist())

Income columns: ['ZIP code', 'average income']
Childcare columns: ['facility_id', 'program_type', 'facility_status', 'facility_name', 'city', 'zip_code', 'school_district_name', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'total_capacity', 'latitude', 'longitude']
Employment columns: ['zipcode', 'employment rate']
Population columns: ['zipcode', 'Total', '-5', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85+']
Potential locations columns: ['zipcode', 'latitude', 'longitude']

After standardization:
Income columns: ['zipcode', 'average income']
Childcare columns: ['facility_id', 'program_type', 'facility_status', 'facility_name', 'city', 'zipcode', 'school_district_name', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'total_capacity', 'latitude', 'longitude']
Employment co

In [14]:
# 3) Force ZIPs to be strings, zero-pad to 5 digits:
dataframes = [avg_individual_income, child_care_regulated, employment_rate, population, potential_locations]
for df in dataframes:
    df['zipcode'] = df['zipcode'].astype(str).str.zfill(5)

# Verify ZIP code formatting
for i, df in enumerate(dataframes, start=1):
    print(f"\nDataFrame {i} ZIP codes sample:", df['zipcode'].head())




DataFrame 1 ZIP codes sample: 0    10001
1    10002
2    10003
3    10004
4    10005
Name: zipcode, dtype: object

DataFrame 2 ZIP codes sample: 0    13323
1    14701
2    12590
3    10960
4    12077
Name: zipcode, dtype: object

DataFrame 3 ZIP codes sample: 0    10001
1    10002
2    10003
3    10004
4    10005
Name: zipcode, dtype: object

DataFrame 4 ZIP codes sample: 0    06390
1    10001
2    10002
3    10003
4    10004
Name: zipcode, dtype: object

DataFrame 5 ZIP codes sample: 0    00501
1    00501
2    00501
3    00501
4    00501
Name: zipcode, dtype: object


In [15]:
# 4) Basic sanity checks:
#    - assert no null ZIPs after padding
#    - preview frames with head() to confirm expected columns

# Check for null values in zipcode columns
print("Checking for null ZIP codes:")
for name, df in zip(['Income', 'Childcare', 'Employment', 'Population', 'Potential Locations'], 
                   [avg_individual_income, child_care_regulated, employment_rate, population, potential_locations]):
    null_count = df['zipcode'].isnull().sum()
    print(f"{name} dataset: {null_count} null ZIP codes")
    # Assert no null values
    assert null_count == 0, f"Found {null_count} null ZIP codes in {name} dataset"


Checking for null ZIP codes:
Income dataset: 0 null ZIP codes
Childcare dataset: 0 null ZIP codes
Employment dataset: 0 null ZIP codes
Population dataset: 0 null ZIP codes
Potential Locations dataset: 0 null ZIP codes


In [16]:
# Analyze childcare facility sizes
# Define size categories based on total capacity
def get_size_category(capacity):
    if capacity <= 100:
        return 'Small'
    elif capacity <= 200:
        return 'Medium'
    elif capacity <= 400:
        return 'Large'
    else:
        return 'XL'

# Add size category column
child_care_regulated['size_category'] = child_care_regulated['total_capacity'].apply(get_size_category)

# Get counts for each category
size_counts = child_care_regulated['size_category'].value_counts().sort_index()
print("Number of facilities in each size category:")
print(size_counts)

# Calculate percentage distribution
size_percentages = (size_counts / len(child_care_regulated) * 100).round(1)
print("\nPercentage distribution:")
for category, percentage in size_percentages.items():
    print(f"{category}: {percentage}%")

# Show summary statistics for each category
print("\nCapacity statistics by category:")
print(child_care_regulated.groupby('size_category')['total_capacity'].agg(['min', 'max', 'mean', 'count']))

Number of facilities in each size category:
size_category
Large       493
Medium     1414
Small     13647
XL           50
Name: count, dtype: int64

Percentage distribution:
Large: 3.2%
Medium: 9.1%
Small: 87.5%
XL: 0.3%

Capacity statistics by category:
               min  max        mean  count
size_category                             
Large          201  400  261.667343    493
Medium         101  200  141.959689   1414
Small            0  100   22.424562  13647
XL             405  890  499.080000     50


In [17]:
# childcare facility -> ZIP
# 1) Treat infant/toddler/preschool as 0-5 supply. school_age as 5-12 supply.
# 2) Compute per-facility:
cap_0_5_fac = child_care_regulated['infant_capacity'] + child_care_regulated['toddler_capacity'] + child_care_regulated['preschool_capacity']
cap_5_12_fac = child_care_regulated['school_age_capacity']
# 3) Aggregate per ZIP:
childcare_zip = child_care_regulated.groupby('zipcode').agg({
       'infant_capacity': 'sum',
       'toddler_capacity': 'sum',
       'preschool_capacity': 'sum',
       'school_age_capacity': 'sum'
   }).reset_index()
# 4) Create ZIP-level totals:
childcare_zip['cap_0_5_zip'] = (cap_0_5_fac := child_care_regulated['infant_capacity'] + 
                                        child_care_regulated['toddler_capacity'] + 
                                        child_care_regulated['preschool_capacity']).groupby(child_care_regulated['zipcode']).sum().values
childcare_zip['cap_5_12_zip'] = (cap_5_12_fac := child_care_regulated['school_age_capacity']).groupby(child_care_regulated['zipcode']).sum().values
childcare_zip['cap_total_zip'] = childcare_zip['cap_0_5_zip'] + childcare_zip['cap_5_12_zip']

childcare_zip.head()

,zipcode,infant_capacity,toddler_capacity,preschool_capacity,school_age_capacity,cap_0_5_zip,cap_5_12_zip,cap_total_zip
0,10001,0,0,0,585,0,585,585
1,10002,0,0,18,4508,18,4508,4526
2,10003,0,0,0,1995,0,1995,1995
3,10004,0,0,0,263,0,263,263
4,10005,0,0,0,39,0,39,39


In [18]:
# Build Population Buckets
# 1) Columns for different age buckets:
#    - under-five column typically named '-5' 
#    - '5-9' and '10-14' used to derive 5–12 population
# 2) Define Buckets: 
pop_0_5 = population['-5']
pop_5_12 = population['5-9'] + 0.5 * population['10-14']  # assumption: half of 10–14 are 10–12
pop_total_0_12 = pop_0_5 + pop_5_12

# 4) Merge these derived cols back into `population` with new names.
population['pop_0_5'] = pop_0_5
population['pop_5_12'] = pop_5_12
population['pop_total_0_12'] = pop_total_0_12
population.head()


,zipcode,Total,-5,5-9,10-14,15-19,20-24,25-29,30-34,35-39,...,55-59,60-64,65-69,70-74,75-79,80-84,85+,pop_0_5,pop_5_12,pop_total_0_12
0,06390,53,0,1,5,0,6,0,9,18,...,0,0,0,0,0,0,0,0,3.5,3.5
1,10001,27004,744,784,942,1035,2296,3806,3588,2524,...,1225,1323,933,815,616,488,576,744,1255.0,1999.0
2,10002,76518,2142,3046,3198,2652,4528,6988,6278,5157,...,6106,4548,4815,4748,2531,2793,2794,2142,4645.0,6787.0
3,10003,53877,1440,1034,953,7013,6344,7100,6427,3221,...,2350,2274,2793,1854,1646,779,1056,1440,1510.5,2950.5
4,10004,4579,433,182,161,108,109,601,724,490,...,279,199,173,2,15,0,0,433,262.5,695.5


In [19]:
# Merge to Master ZIP Table 

# 1) Start from population (ensures we keep only ZIPs with population data):
master = population[['zipcode', 'pop_0_5', 'pop_5_12', 'pop_total_0_12']].copy()
#
# 2) Merge income and employment:
master = master.merge(avg_individual_income[['zipcode','average income']], on='zipcode', how='left')
master = master.merge(employment_rate[['zipcode','employment rate']], on='zipcode', how='left')
#    - Rename columns after merge: 'average income' -> 'avg_income', etc.
#
# 3) Merge existing capacities by ZIP:
master = master.merge(childcare_zip[['zipcode','cap_0_5_zip','cap_5_12_zip','cap_total_zip']], 
                        on='zipcode', how='left')
#    - Fill missing capacities with 0 for ZIPs without facilities.
#
# 4) Sanity checks:
#    - No negative values
#    - Reasonable ranges (income order of magnitude, employment rate in [0,1])
#    - Count how many ZIPs have 0 capacity but positive population
master.sample(n = 20)

,zipcode,pop_0_5,pop_5_12,pop_total_0_12,average income,employment rate,cap_0_5_zip,cap_5_12_zip,cap_total_zip
908,12827,329,205.0,534.0,63092.105263,0.517288,NaN,NaN,NaN
1624,14710,106,388.0,494.0,63370.253165,0.505600,NaN,NaN,NaN
354,11433,2837,3679.0,6516.0,44920.749280,0.446417,0.0,1865.0,1865.0
863,12762,0,11.5,11.5,62980.769231,0.742857,24.0,0.0,24.0
187,10803,656,1910.0,2566.0,118323.627288,0.452458,559.0,101.0,660.0
267,11105,2103,2427.5,4530.5,68860.908142,0.494375,22.0,760.0,782.0
1099,13132,126,245.5,371.5,65040.106952,0.535050,0.0,2.0,2.0
677,12304,1310,1917.5,3227.5,52301.054650,0.476234,408.0,168.0,576.0
1135,13209,917,905.0,1822.0,57932.692308,0.513249,75.0,122.0,197.0
711,12435,7,29.0,36.0,65441.176471,0.735931,NaN,NaN,NaN


# High-Demand Classification & Policy Thresholds


 High-demand if employment_rate ≥ 0.60 or avg_income ≤ 60,000.

Coverage requirements:

– High: cap_total_zip + new_total_zip ≥ 0.5 * pop_total_0_12

– Normal: cap_total_zip + new_total_zip ≥ 1/3 * pop_total_0_12

0–5 policy: cap_0_5_zip + new_0_5_zip ≥ (2/3) * pop_0_5.

We compute the binary flag now for later use in constraints.

In [20]:
#Compute High-Demand Flag 

# 1) Create 'is_high_demand' as int {0,1}:
#    - Ensure employment_rate ∈ [0,1]; if your file stores percentages (e.g., 62 means 62%), divide by 100.
#    - Ensure income in dollars (no thousands scaling).
master['is_high_demand'] = ((master['employment rate'] >= 0.60) | 
                            (master['average income'] <= 60000)).astype(int)


# 2) Diagnostics:
#    - how many ZIPs are high-demand vs normal
#    - inspect boundary cases near 0.60 employment and $60k income
master['is_high_demand'].value_counts()


is_high_demand
0    1085
1     561
Name: count, dtype: int64

# Candidate Locations & Distance Constraints

The realistic stage requires a minimum 0.06 miles between any two facilities (existing or new) within each area/ZIP (as presented). We will:

Clean/validate coordinates in child_care_regulated.csv (existing facilities) and potential_locations.csv (candidate sites).

Build conflict pairs (i, j) within each ZIP where distance < 0.06, then forbid simultaneous selection.

*Notes:*

If existing facilities lack coordinates, impute with ZIP centroids or drop the distance constraint for those ZIPs (document the choice).

Use great-circle (Haversine) distance for accuracy.

In [21]:
# Distance Matrix & Conflict Pairs 

# 1) Prepare coordinates:
#    - From childcare: keep rows with non-null lat/lon → 'existing_sites' with ['zipcode','lat','lon','facility_id']
#    - From locations: keep rows with non-null lat/lon → 'candidate_sites' with ['zipcode','lat','lon','site_id']
#    - Optional: compute ZIP centroids if many NaNs (document the imputation).
existing_sites = child_care_regulated[['zipcode', 'latitude', 'longitude', 'facility_id']].dropna(subset=['latitude', 'longitude'])
existing_sites = existing_sites.rename(columns={'latitude': 'lat', 'longitude': 'lon'})
# Ensure candidate sites have a unique site_id
if 'site_id' not in potential_locations.columns:
    potential_locations['site_id'] = potential_locations.index.astype(str)

candidate_sites = potential_locations[['zipcode', 'latitude', 'longitude', 'site_id']].dropna(subset=['latitude', 'longitude'])
candidate_sites = candidate_sites.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

# 2) Define a function haversine(lat1, lon1, lat2, lon2) → miles.
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Radius of Earth in miles
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

# 3) Build conflict sets within each ZIP:
#    (A) Between new sites: for each pair (i,j) in candidate_sites[zipcode], if distance < 0.06 → add (i,j) to C_newnew[z]
#    (B) Between new and existing: for each new i and existing f in same ZIP, if dist < 0.06 → add (i,f) to C_newold[z]
conflict_threshold = 0.06  # miles
new_new_conflicts = {}
new_old_conflicts = {}

# (A) Between new sites: for each pair (i,j) in candidate_sites[zipcode], if distance < 0.06 → add (i,j) to C_newnew[z]
for zipcode, group in candidate_sites.groupby('zipcode'):
    pairs = []
    sites = group.reset_index(drop=True)
    for i in range(len(sites)):
        for j in range(i+1, len(sites)):
            dist = haversine(sites.loc[i, 'lat'], sites.loc[i, 'lon'], sites.loc[j, 'lat'], sites.loc[j, 'lon'])
            if dist < conflict_threshold:
                pairs.append((sites.loc[i, 'site_id'], sites.loc[j, 'site_id']))
    if pairs:
        new_new_conflicts[zipcode] = pairs

# (B) Between new and existing: for each new i and existing f in same ZIP, if dist < 0.06 → add (i,f) to C_newold[z]
for zipcode, new_group in candidate_sites.groupby('zipcode'):
    old_group = existing_sites[existing_sites['zipcode'] == zipcode]
    pairs = []
    for i in range(len(new_group)):
        for j in range(len(old_group)):
            dist = haversine(new_group.iloc[i]['lat'], new_group.iloc[i]['lon'], old_group.iloc[j]['lat'], old_group.iloc[j]['lon'])
            if dist < conflict_threshold:
                pairs.append((new_group.iloc[i]['site_id'], old_group.iloc[j]['facility_id']))
    if pairs:
        new_old_conflicts[zipcode] = pairs

print('New-New Conflicts by ZIP:', {k: len(v) for k, v in new_new_conflicts.items()})
print('New-Old Conflicts by ZIP:', {k: len(v) for k, v in new_old_conflicts.items()})

New-New Conflicts by ZIP: {'00501': 37, '00544': 30, '06390': 29, '10001': 39, '10002': 32, '10003': 36, '10004': 49, '10005': 34, '10006': 43, '10007': 32, '10008': 37, '10009': 35, '10010': 40, '10011': 43, '10012': 33, '10013': 42, '10014': 33, '10016': 37, '10017': 31, '10018': 37, '10019': 36, '10020': 30, '10021': 39, '10022': 36, '10023': 39, '10024': 41, '10025': 42, '10026': 34, '10027': 48, '10028': 42, '10029': 37, '10030': 32, '10031': 37, '10032': 34, '10033': 35, '10034': 39, '10035': 30, '10036': 29, '10037': 41, '10038': 36, '10039': 36, '10040': 47, '10041': 37, '10043': 42, '10044': 38, '10045': 38, '10055': 42, '10060': 45, '10065': 45, '10069': 36, '10075': 27, '10080': 39, '10081': 39, '10087': 44, '10090': 33, '10101': 33, '10102': 32, '10103': 35, '10104': 33, '10105': 40, '10106': 41, '10107': 33, '10108': 44, '10109': 31, '10110': 43, '10111': 39, '10112': 35, '10113': 43, '10114': 31, '10115': 38, '10116': 31, '10117': 33, '10118': 29, '10119': 38, '10120': 44

# Expansion Limits & Piecewise Costs (Facility-Level)

For each existing facility f with current capacity nf:

Max expansion x_f ≤ min(500, 0.2 * nf).

Piecewise cost by expansion ratio x_f / nf:

0–10%: unit cost = (20,000 + 200 * nf) / nf

10–15%: unit cost = (20,000 + 400 * nf) / nf

15–20%: unit cost = (20,000 + 1,000 * nf) / nf

We linearize with three segment variables x_f1, x_f2, x_f3 s.t.
x_f = x_f1 + x_f2 + x_f3 and segment caps enforce the correct ranges.

Also remember $100 per new 0–5 slot surcharge, applied to:

0–5 portion of expansion (x_f_0_5) and

0–5 portion in new facilities.

In [34]:
# Build Facility-Level Parameters for Expansion

# 1) Compute total and 0-5 capacity per facility
child_care_regulated['nf'] = child_care_regulated['infant_capacity'] + child_care_regulated['toddler_capacity'] + child_care_regulated['preschool_capacity'] + child_care_regulated['school_age_capacity']
child_care_regulated['nf_0_5'] = child_care_regulated['infant_capacity'] + child_care_regulated['toddler_capacity'] + child_care_regulated['preschool_capacity']

# 2) Segment caps and overall cap
child_care_regulated['s1_max'] = 0.10 * child_care_regulated['nf']
child_care_regulated['s2_max'] = 0.05 * child_care_regulated['nf']
child_care_regulated['s3_max'] = 0.05 * child_care_regulated['nf']
child_care_regulated['overall_cap'] = child_care_regulated[['nf']].apply(lambda x: min(500, 0.20 * x['nf']), axis=1)

# 3) Unit costs per slot for each segment
child_care_regulated['c1'] = (20000 + 200 * child_care_regulated['nf']) / child_care_regulated['nf']
child_care_regulated['c2'] = (20000 + 400 * child_care_regulated['nf']) / child_care_regulated['nf']
child_care_regulated['c3'] = (20000 + 1000 * child_care_regulated['nf']) / child_care_regulated['nf']

# 4) Store as dictionaries keyed by facility_id
# Remove facilities with nf == 0 or any cost coefficient is inf or NaN
mask_valid = (
    (child_care_regulated['nf'] > 0) &
    np.isfinite(child_care_regulated['c1']) &
    np.isfinite(child_care_regulated['c2']) &
    np.isfinite(child_care_regulated['c3'])
)
child_care_regulated_valid = child_care_regulated[mask_valid].copy()

# Rebuild facility_params using only valid facilities
facility_params = {}
for _, row in child_care_regulated_valid.iterrows():
    facility_params[row['facility_id']] = {
        'zipcode': row['zipcode'],
        'nf': row['nf'],
        'nf_0_5': row['nf_0_5'],
        's1_max': row['s1_max'],
        's2_max': row['s2_max'],
        's3_max': row['s3_max'],
        'overall_cap': row['overall_cap'],
        'c1': row['c1'],
        'c2': row['c2'],
        'c3': row['c3']
    }

print('Sample facility parameters:', list(facility_params.items())[:3])

Sample facility parameters: [(2416, {'zipcode': '13323', 'nf': 2, 'nf_0_5': 0, 's1_max': 0.2, 's2_max': 0.1, 's3_max': 0.1, 'overall_cap': 0.4, 'c1': 10200.0, 'c2': 10400.0, 'c3': 11000.0}), (5555, {'zipcode': '14701', 'nf': 2, 'nf_0_5': 0, 's1_max': 0.2, 's2_max': 0.1, 's3_max': 0.1, 'overall_cap': 0.4, 'c1': 10200.0, 'c2': 10400.0, 'c3': 11000.0}), (9066, {'zipcode': '12590', 'nf': 2, 'nf_0_5': 0, 's1_max': 0.2, 's2_max': 0.1, 's3_max': 0.1, 'overall_cap': 0.4, 'c1': 10200.0, 'c2': 10400.0, 'c3': 11000.0})]


In [35]:
# Check for zero values in key columns
print('Number of facilities with nf == 0:', (child_care_regulated['nf'] == 0).sum())
print('Number of facilities with nf_0_5 == 0:', (child_care_regulated['nf_0_5'] == 0).sum())
print('Number of facilities with s1_max == 0:', (child_care_regulated['s1_max'] == 0).sum())
print('Number of facilities with s2_max == 0:', (child_care_regulated['s2_max'] == 0).sum())
print('Number of facilities with s3_max == 0:', (child_care_regulated['s3_max'] == 0).sum())
print('Number of facilities with overall_cap == 0:', (child_care_regulated['overall_cap'] == 0).sum())
print('Number of facilities with c1 == 0:', (child_care_regulated['c1'] == 0).sum())
print('Number of facilities with c2 == 0:', (child_care_regulated['c2'] == 0).sum())
print('Number of facilities with c3 == 0:', (child_care_regulated['c3'] == 0).sum())
print('Number of facilities with c1 == inf:', np.isinf(child_care_regulated['c1']).sum())
print('Number of facilities with c2 == inf:', np.isinf(child_care_regulated['c2']).sum())
print('Number of facilities with c3 == inf:', np.isinf(child_care_regulated['c3']).sum())

Number of facilities with nf == 0: 815
Number of facilities with nf_0_5 == 0: 13498
Number of facilities with s1_max == 0: 815
Number of facilities with s2_max == 0: 815
Number of facilities with s3_max == 0: 815
Number of facilities with overall_cap == 0: 815
Number of facilities with c1 == 0: 0
Number of facilities with c2 == 0: 0
Number of facilities with c3 == 0: 0
Number of facilities with c1 == inf: 815
Number of facilities with c2 == inf: 815
Number of facilities with c3 == inf: 815


In [36]:
# Fill NaN values in ZIP-level capacity columns and cost/cap columns to prevent optimization errors
master['cap_total_zip'] = master['cap_total_zip'].fillna(0)
master['cap_0_5_zip'] = master['cap_0_5_zip'].fillna(0)
childcare_zip = childcare_zip.fillna(0)
# Fill NaN in cost/cap columns for facilities
for col in ['c1', 'c2', 'c3', 's1_max', 's2_max', 's3_max', 'overall_cap']:
    child_care_regulated[col] = child_care_regulated[col].fillna(0)

In [37]:
# List facilities with nf == 0 (most critical)
print(child_care_regulated[child_care_regulated['nf'] == 0][['facility_id', 'zipcode', 'nf', 'nf_0_5', 's1_max', 'c1', 'c2','c3']])

# List facilities with any cost coefficient == 0
for col in ['c1', 'c2', 'c3']:
    print(f"Facilities with {col} == 0:")
    print(child_care_regulated[child_care_regulated[col] == 0][['facility_id', 'zipcode', 'nf', col]])

       facility_id zipcode  nf  nf_0_5  s1_max   c1   c2   c3
6            51349   10474   0       0     0.0  inf  inf  inf
10          126425   10465   0       0     0.0  inf  inf  inf
21          274852   10460   0       0     0.0  inf  inf  inf
25          349917   10452   0       0     0.0  inf  inf  inf
28          386539   10451   0       0     0.0  inf  inf  inf
...            ...     ...  ..     ...     ...  ...  ...  ...
15553       897247   10302   0       0     0.0  inf  inf  inf
15558       902343   10470   0       0     0.0  inf  inf  inf
15572       905273   10451   0       0     0.0  inf  inf  inf
15573       894974   10462   0       0     0.0  inf  inf  inf
15600       897263   11205   0       0     0.0  inf  inf  inf

[815 rows x 8 columns]
Facilities with c1 == 0:
Empty DataFrame
Columns: [facility_id, zipcode, nf, c1]
Index: []
Facilities with c2 == 0:
Empty DataFrame
Columns: [facility_id, zipcode, nf, c2]
Index: []
Facilities with c3 == 0:
Empty DataFrame
Columns: 

In [38]:
# Check for 0, NaN, or Inf in key facility columns
import numpy as np
cols_to_check = ['nf', 'nf_0_5', 's1_max', 's2_max', 's3_max', 'overall_cap', 'c1', 'c2', 'c3']
for col in cols_to_check:
    print(f"\nFacilities with {col} == 0:")
    print(child_care_regulated[child_care_regulated[col] == 0][['facility_id', 'zipcode', col]])
    print(f"Facilities with {col} as NaN:")
    print(child_care_regulated[child_care_regulated[col].isna()][['facility_id', 'zipcode', col]])
    print(f"Facilities with {col} == Inf:")
    print(child_care_regulated[np.isinf(child_care_regulated[col])][['facility_id', 'zipcode', col]])


Facilities with nf == 0:
       facility_id zipcode  nf
6            51349   10474   0
10          126425   10465   0
21          274852   10460   0
25          349917   10452   0
28          386539   10451   0
...            ...     ...  ..
15553       897247   10302   0
15558       902343   10470   0
15572       905273   10451   0
15573       894974   10462   0
15600       897263   11205   0

[815 rows x 3 columns]
Facilities with nf as NaN:
Empty DataFrame
Columns: [facility_id, zipcode, nf]
Index: []
Facilities with nf == Inf:
Empty DataFrame
Columns: [facility_id, zipcode, nf]
Index: []

Facilities with nf_0_5 == 0:
       facility_id zipcode  nf_0_5
0             2416   13323       0
1             5555   14701       0
2             9066   12590       0
4            41016   12077       0
6            51349   10474       0
...            ...     ...     ...
15599       892735   10462       0
15600       897263   11205       0
15601       901966   10705       0
15602       892455  

# New Facility Options (Discrete Sizes)

We allow discrete choices per candidate site:

Small: 100 total slots; max 50 in 0–5; cost $65,000.

Medium: 200 total slots; max 100 in 0–5; cost $95,000.

Large: 400 total slots; max 200 in 0–5; cost $115,000.

Decision variables:

Binary y[s,k] = 1 if we build size k ∈ {S, M, L} at candidate site s (enforce ≤ 1 size per site).

Integer/continuous new05[s] for 0–5 slots at site s, bounded by the chosen size’s 0–5 cap.

Derived newtotal[s] by size (100/200/400).

In [39]:
# New Facility Parameters (comments only)

# 1) Define size sets and parameters:
sizes = {'S': {'tot':100, 'max05':50,  'cost':65000},
            'M': {'tot':200, 'max05':100, 'cost':95000},
            'L': {'tot':400, 'max05':200, 'cost':115000}}

# 2) Prepare candidate site parameter dictionaries
candidate_params = {}
for _, row in candidate_sites.iterrows():
    candidate_params[row['site_id']] = {
        'zipcode': row['zipcode'],
        'lat': row['lat'],
        'lon': row['lon'],
        'sizes': sizes
    }

# 3) Example: For optimization model setup (pseudo-code, not actual Gurobi code)
# For each candidate site s:
#   - Decision variables: y_s_S, y_s_M, y_s_L (binaries)
#   - Constraint: y_s_S + y_s_M + y_s_L <= 1
#   - newtotal_s = 100*y_s_S + 200*y_s_M + 400*y_s_L
#   - new05_s <= 50*y_s_S + 100*y_s_M + 200*y_s_L
#   - Equipment surcharge: 100 * (new05_s + sum_f x_f_0_5 for same ZIP)

# 4) Map each site to its ZIP for coverage aggregation
site_zip_map = {row['site_id']: row['zipcode'] for _, row in candidate_sites.iterrows()}

print('Sample candidate site parameters:', list(candidate_params.items())[:3])
print('Site to ZIP mapping:', list(site_zip_map.items())[:3])



Sample candidate site parameters: [('0', {'zipcode': '00501', 'lat': 40.81637627007854, 'lon': -73.04079621267256, 'sizes': {'S': {'tot': 100, 'max05': 50, 'cost': 65000}, 'M': {'tot': 200, 'max05': 100, 'cost': 95000}, 'L': {'tot': 400, 'max05': 200, 'cost': 115000}}}), ('1', {'zipcode': '00501', 'lat': 40.81745526752143, 'lon': -73.04420233634006, 'sizes': {'S': {'tot': 100, 'max05': 50, 'cost': 65000}, 'M': {'tot': 200, 'max05': 100, 'cost': 95000}, 'L': {'tot': 400, 'max05': 200, 'cost': 115000}}}), ('2', {'zipcode': '00501', 'lat': 40.81387309598677, 'lon': -73.04218211773868, 'sizes': {'S': {'tot': 100, 'max05': 50, 'cost': 65000}, 'M': {'tot': 200, 'max05': 100, 'cost': 95000}, 'L': {'tot': 400, 'max05': 200, 'cost': 115000}}})]
Site to ZIP mapping: [('0', '00501'), ('1', '00501'), ('2', '00501')]


# Optimization Model (Variables, Objective, Constraints)



## Decision Variables

Expansion per facility: x_f1, x_f2, x_f3 ≥ 0 (segment expansions); x_f = x_f1 + x_f2 + x_f3.

0–5 expansion portion per facility: x_f_0_5 ≥ 0, with x_f_0_5 ≤ x_f.

New facility binaries: y[s,k] ∈ {0,1} (size selection).

New 0–5 slots per site: new05[s] ≥ 0, bounded by chosen size.

(Optionally) new total per site newtotal[s] as expression from y.

## Objective
Minimize:

Sum of expansion costs over segments (c1x_f1 + c2x_f2 + c3*x_f3),

Plus construction costs (∑_s ∑_k cost_k * y[s,k]),

Plus 0–5 surcharge: 100 * ( ∑_f x_f_0_5 + ∑_s new05[s] ).

## Constraints

Segment caps:
x_f1 ≤ s1_max, x_f2 ≤ s2_max, x_f3 ≤ s3_max and x_f ≤ overall_cap_f.

0–5 expansion subset: 0 ≤ x_f_0_5 ≤ x_f.

Site size selection: ∑_k y[s,k] ≤ 1.

0–5 allocation in new sites: new05[s] ≤ ∑_k max05_k * y[s,k].

Coverage per ZIP (choose formula based on high/normal):
existing_total_zip + (∑_f in ZIP x_f) + (∑_s in ZIP newtotal[s]) ≥ threshold_zip * pop_total_0_12
where threshold_zip is 0.5 if high-demand else 1/3.

0–5 policy per ZIP:
existing_0_5_zip + (∑_f in ZIP x_f_0_5) + (∑_s in ZIP new05[s]) ≥ (2/3) * pop_0_5.

### Distance constraints:
For any conflict pair (s, s’) in same ZIP with dist < 0.06, forbid simultaneous selection:
∑_k y[s,k] + ∑_k y[s’,k] ≤ 1.
For new vs existing conflicts, either (a) forbid site s, or (b) ensure no build at s if too close to an existing facility.

### Domains: binaries and non-negativity as defined.

In [40]:
# Gurobi Model: Full Implementation
m = gp.Model("childcare_deserts")

# --- VARIABLES ---
# Facility expansion variables (piecewise segments)
x_f1 = {}  # 0-10% expansion
x_f2 = {}  # 10-15% expansion
x_f3 = {}  # 15-20% expansion
x_f_0_5 = {}  # 0-5 portion of expansion
for f, params in facility_params.items():
    x_f1[f] = m.addVar(lb=0, ub=params['s1_max'], name=f"x_f1_{f}")
    x_f2[f] = m.addVar(lb=0, ub=params['s2_max'], name=f"x_f2_{f}")
    x_f3[f] = m.addVar(lb=0, ub=params['s3_max'], name=f"x_f3_{f}")
    x_f_0_5[f] = m.addVar(lb=0, ub=params['overall_cap'], name=f"x_f_0_5_{f}")

# New facility variables
y = {}  # y[s,k]: binary for site s, size k
new05 = {}  # new 0-5 slots at site s
for s, params in candidate_params.items():
    y[s] = {}
    for k in sizes.keys():
        y[s][k] = m.addVar(vtype=GRB.BINARY, name=f"y_{s}_{k}")
    new05[s] = m.addVar(lb=0, ub=max([sizes[k]['max05'] for k in sizes]), name=f"new05_{s}")

# --- OBJECTIVE ---
expansion_cost = gp.quicksum(facility_params[f]['c1'] * x_f1[f] + facility_params[f]['c2'] * x_f2[f] + facility_params[f]['c3'] * x_f3[f] for f in facility_params)
construction_cost = gp.quicksum(sizes[k]['cost'] * y[s][k] for s in y for k in y[s])
equipment_cost = 100 * (gp.quicksum(x_f_0_5[f] for f in x_f_0_5) + gp.quicksum(new05[s] for s in new05))
m.setObjective(expansion_cost + construction_cost + equipment_cost, GRB.MINIMIZE)

# --- CONSTRAINTS ---
# Facility segment caps and total cap
for f, params in facility_params.items():
    m.addConstr(x_f1[f] + x_f2[f] + x_f3[f] <= params['overall_cap'], name=f"total_cap_{f}")
    m.addConstr(x_f_0_5[f] <= x_f1[f] + x_f2[f] + x_f3[f], name=f"x_f_0_5_leq_total_{f}")

# Site size selection and new05 allocation
for s in y:
    m.addConstr(gp.quicksum(y[s][k] for k in y[s]) <= 1, name=f"site_size_select_{s}")
    m.addConstr(new05[s] <= gp.quicksum(sizes[k]['max05'] * y[s][k] for k in y[s]), name=f"new05_max_{s}")

# --- ZIP-level coverage and 0-5 policy ---
for idx, row in master.iterrows():
    zipc = row['zipcode']
    # Coverage threshold: high-demand ZIPs get 0.5, else 1/3
    threshold = 0.5 if row['is_high_demand'] == 1 else (1/3)
    # Existing totals
    existing_total_zip = row['cap_total_zip'] if not np.isnan(row['cap_total_zip']) else 0
    existing_0_5_zip = row['cap_0_5_zip'] if not np.isnan(row['cap_0_5_zip']) else 0
    # Expansion in ZIP
    expansion_total_zip = gp.quicksum(x_f1[f] + x_f2[f] + x_f3[f] for f, params in facility_params.items() if params['zipcode'] == zipc)
    expansion_0_5_zip = gp.quicksum(x_f_0_5[f] for f, params in facility_params.items() if params['zipcode'] == zipc)
    # New facilities in ZIP
    newtotal_zip = gp.quicksum(gp.quicksum(sizes[k]['tot'] * y[s][k] for k in y[s]) for s in y if site_zip_map[s] == zipc)
    new05_zip = gp.quicksum(new05[s] for s in new05 if site_zip_map[s] == zipc)
    # Coverage constraint
    m.addConstr(existing_total_zip + expansion_total_zip + newtotal_zip >= threshold * row['pop_total_0_12'], name=f"coverage_zip_{zipc}")
    # 0-5 policy constraint
    m.addConstr(existing_0_5_zip + expansion_0_5_zip + new05_zip >= (2/3) * row['pop_0_5'], name=f"policy_0_5_zip_{zipc}")

# --- Distance conflict constraints ---
# (A) New-new conflicts: for each ZIP, for each pair (s1, s2)
for zipc, pairs in new_new_conflicts.items():
    for s1, s2 in pairs:
        m.addConstr(gp.quicksum(y[s1][k] for k in y[s1]) + gp.quicksum(y[s2][k] for k in y[s2]) <= 1, name=f"conflict_newnew_{zipc}_{s1}_{s2}")
# (B) New-old conflicts: for each ZIP, for each (s, f)
for zipc, pairs in new_old_conflicts.items():
    for s, f in pairs:
        # Option: forbid building at s if too close to existing f
        m.addConstr(gp.quicksum(y[s][k] for k in y[s]) <= 1 - 1, name=f"conflict_newold_{zipc}_{s}_{f}")  # Effectively y[s,k] == 0

# --- Solve ---
m.optimize()

# --- Results extraction (see next cell for details)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M3 Max
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

# Outputs & Diagnostics

Key totals: minimum total funding, # of new facilities (by size), total expansions.

Per-ZIP results: whether threshold met, additional slots added, cost breakdown.

Feasibility checks: any ZIPs failing constraints; binding constraints insights.

In [ ]:
# Postprocessing 

# 1) Extract solution values:
#    - For each facility: x_f1, x_f2, x_f3, x_f_0_5; compute x_f_total = sum segments
#    - For each site: chosen size y[s,k], new05[s], newtotal[s]
#
# 2) Aggregate back to ZIP:
#    - added_total_zip = sum_f_in_zip x_f_total + sum_s_in_zip newtotal[s]
#    - added_0_5_zip = sum_f_in_zip x_f_0_5 + sum_s_in_zip new05[s]
#    - final_total_zip = existing_total_zip + added_total_zip
#    - final_0_5_zip = existing_0_5_zip + added_0_5_zip
#
# 3) Verify constraints:
#    - For each ZIP, check coverage threshold and 0–5 policy
#    - Flag any ZIPs that are exactly at threshold (potential sensitivity)
#
# 4) Cost breakdown:
#    - expansion_cost = sum_f (c1*x_f1 + c2*x_f2 + c3*x_f3)
#    - construction_cost = sum_s,k cost_k * y[s,k]
#    - equipment_cost = 100 * (sum_f x_f_0_5 + sum_s new05[s])
#    - total_cost = expansion + construction + equipment